<a href="https://colab.research.google.com/github/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/blob/main/my%20training%20code/step4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 💡 徹底拔除無關的影音干擾套件，回歸純文字 NLP 環境
!pip uninstall torchcodec torchvision -y

Found existing installation: torchcodec 0.10.0+cu128
Uninstalling torchcodec-0.10.0+cu128:
  Successfully uninstalled torchcodec-0.10.0+cu128
Found existing installation: torchvision 0.27.0
Uninstalling torchvision-0.27.0:
  Successfully uninstalled torchvision-0.27.0


In [2]:
# 💡 專門修復環境相容性的獨立儲存格，這輩子只需要 Run 這一次！
!pip install --upgrade "transformers==4.46.0" setfit datasets torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.1 MB/s eta 0:00:00
  Using cached torchvision-0.27.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 99.8 MB/s eta 0:00:00
Using cached torchvision-0.27.0-cp312-cp312-manylinux_2_28_x86_64.whl (7.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 109.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
   

In [1]:
import transformers
import sys

# ==========================================
# 🌟 終極黑客動態修補線 (安全防重複版)
# 內建 hasattr 偵測鎖，徹底根除 RecursionError 鏡像地獄！
# ==========================================
if not hasattr(transformers.Trainer, "_is_patched"):
    # 🚨 改成 lambda 函數，滿足 transformers 內部的呼叫機制 ()
    transformers.training_args.default_logdir = lambda: "tmp_trainer"

    # 智慧過濾攔截器
    original_trainer_init = transformers.Trainer.__init__
    def patched_trainer_init(self, *args, **kwargs):
        kwargs.pop("optimizer_cls_and_kwargs", None)  # 智慧型過濾
        return original_trainer_init(self, *args, **kwargs)

    transformers.Trainer.__init__ = patched_trainer_init
    transformers.Trainer._is_patched = True  # 🔓 成功打上安全標記，防止二次注入
    print("🔒 智慧引導安全修補線：配置成功！")
else:
    print("🔓 偵測到環境已修補，自動跳過以防範 RecursionError。")

# ==========================================
# 正式導入其餘必要元件
# ==========================================
import pandas as pd
import torch
import time
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
import urllib.request
import yaml
import re

print("🚀 啟動階段四：SetFit 意圖解纏與模型微調 (全遠端智慧對齊黑客版)")

# ==========================================
# 0. 載入即時推論的防禦設定 (地名與豁免詞)
# ==========================================
def get_sanitizer(yaml_url):
    response = urllib.request.urlopen(yaml_url)
    config = yaml.safe_load(response)
    loc_config = config.get('locations', {})
    all_locations = []
    for cat in loc_config:
        all_locations.extend(loc_config[cat])
    if not all_locations: return None
    return re.compile(f"({'|'.join(all_locations)})")

YAML_URL = "https://raw.githubusercontent.com/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/main/training%20data/keywords.yaml"
city_pattern = get_sanitizer(YAML_URL)

def preprocess_texts(texts, replacement="[LOC]"):
    if not city_pattern: return texts
    if isinstance(texts, str): return city_pattern.sub(replacement, texts)
    return [city_pattern.sub(replacement, str(t)) if pd.notna(t) else "" for t in texts]

IGNORE_URL = "https://raw.githubusercontent.com/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/main/training%20data/ignore.txt"
ignore_response = urllib.request.urlopen(IGNORE_URL)
ignore_keywords = [line.decode('utf-8').strip() for line in ignore_response if line.strip() and not line.decode('utf-8').startswith('#')]


# ==========================================
# 1. 載入三方資料（全遠端雲端化 + 智慧防禦欄位對齊）
# ==========================================
print("\n正在從遠端 GitHub 載入訓練資料集...")

GOLD_URL = "https://media.githubusercontent.com/media/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/refs/heads/main/training%20data/gold_candidates_review_with_ignore.csv"
NORMAL_URL = "https://media.githubusercontent.com/media/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/refs/heads/main/training%20data/normal_pool_sample.csv"
SNORKEL_URL = "https://media.githubusercontent.com/media/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/refs/heads/main/training%20data/snorkel_labeled_training_data.csv"

def dynamic_column_aligner(df, dataset_name):
    if 'text' in df.columns:
        return df
    elif 'name' in df.columns:
        return df.rename(columns={'name': 'text'})
    else:
        first_col = df.columns[0]
        return df.rename(columns={first_col: 'text'})

# A. 載入黃金樣本
df_gold = pd.read_csv(GOLD_URL)
df_gold = dynamic_column_aligner(df_gold, "Gold 黃金樣本")
df_gold = df_gold.dropna(subset=['text'])
df_gold = df_gold.rename(columns={'proposed_category': 'snorkel_label'})

label_translation = {
    'loan': '借貸融資',
    'porn': '黃色與特種行業',
    'gambling': '博弈',
    'scam_recruitment': '詐騙高風險招募',
    'drugs': '毒品'
}
df_gold['snorkel_label'] = df_gold['snorkel_label'].map(label_translation)

# B. 載入 Snorkel 資料
df_snorkel = pd.read_csv(SNORKEL_URL)
df_snorkel = dynamic_column_aligner(df_snorkel, "Snorkel 弱標籤資料")

# C. 載入正常背景資料
df_normal = pd.read_csv(NORMAL_URL)
df_normal = dynamic_column_aligner(df_normal, "Normal 背景資料")
df_normal['snorkel_label'] = "正常困難負樣本"

# --- 關鍵：資料整合與去重 ---
df_snorkel_filtered = df_snorkel[~df_snorkel['text'].isin(df_gold['text'])].copy()
df_train_full = pd.concat([df_gold, df_snorkel_filtered, df_normal], ignore_index=True)

# 因為毒品只有 2 筆，強制移除避免干擾大模型微調
df_train_full = df_train_full[df_train_full['snorkel_label'] != "毒品"].copy()
df_gold = df_gold[df_gold['snorkel_label'] != "毒品"].copy()

print("\n📊 [降採樣前] 混合訓練資料分佈：")
print(df_train_full['snorkel_label'].value_counts())


# ==========================================
# 2. 執行「精銳降採樣」策略
# ==========================================
MAX_SAMPLES_PER_CLASS = 150

sampled_chunks = []
for label, group in df_train_full.groupby('snorkel_label'):
    if len(group) <= MAX_SAMPLES_PER_CLASS:
        sampled_chunks.append(group)
    else:
        sampled_chunks.append(group.sample(n=MAX_SAMPLES_PER_CLASS, random_state=42))

df_train = pd.concat(sampled_chunks, ignore_index=True)
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n🎯 [降採樣後] SetFit 訓練分佈 (上限 {MAX_SAMPLES_PER_CLASS} 筆/類別)：")
print(df_train['snorkel_label'].value_counts())

# ------------------------------------------------
unique_labels = sorted(df_train['snorkel_label'].unique())
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

print(f"\n訓練類別映射表: {label_to_id}")

df_train['label'] = df_train['snorkel_label'].map(label_to_id)
train_dataset = Dataset.from_pandas(df_train[['text', 'label']])


# ==========================================
# 3. 準備黃金驗證集
# ==========================================
eval_df = df_gold.copy()
eval_df['label'] = eval_df['snorkel_label'].map(label_to_id)
eval_dataset = Dataset.from_pandas(eval_df[['text', 'label']])


# ==========================================
# 4. 載入基礎模型與硬體加速
# ==========================================
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n正在載入多語系基礎模型，使用加速裝置: {device}")

model = SetFitModel.from_pretrained(
    "paraphrase-multilingual-MiniLM-L12-v2",
    labels=list(label_to_id.keys())
)
model.to(device)


# ==========================================
# 5. 設定訓練參數與啟動 Trainer
# ==========================================
args = TrainingArguments(
    batch_size=64,
    num_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
    num_iterations=10
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    metric="accuracy"
)

print("\n🔥 開始進行對比學習微調 (含黃金樣本強化)...")
start_time = time.time()
trainer.train()
print(f"\n✅ 模型訓練完成！總耗時: {time.time() - start_time:.2f} 秒")


# ==========================================
# 6. 儲存模型與評估
# ==========================================
model_path = "./my_intent_classifier_setfit"
model.save_pretrained(model_path)
print(f"🎉 強化版模型已儲存至: {model_path}")

eval_results = trainer.evaluate()
print(f"\n📊 黃金樣本最終測試成績 (Accuracy): {eval_results['accuracy']:.4f}")


# ==========================================
# 7. 實戰預測測試 (地名脫敏 + 豁免白名單雙重防禦)
# ==========================================
print("\n--- 🧠 實戰預測測試 (驗證防禦效果) ---")

raw_test_sentences = [
    "日本東京灣的萬豪酒店住宿體驗分享",
    "全額贷，無勞保也可強力過件，當日放款",
    "群組受害者自救會，律師免費諮詢幫你拿回資金",
    "註冊首存送 1000，今晚百家樂帶你飛",
    "誠徵台北外派公關，免經驗日領萬元",
    "台南信貸，手續簡便當天撥款",
    "八大"
]

sanitized_sentences = preprocess_texts(raw_test_sentences)

for raw, clean in zip(raw_test_sentences, sanitized_sentences):
    print(f"原始: {raw}")
    if raw != clean:
        print(f"清洗: {clean}")

    if any(ignore_word in raw for ignore_word in ignore_keywords):
         print(f"👉 判定結果: [正常困難負樣本] (因觸發豁免詞彙)")
    else:
         label_id = model.predict([clean])[0]
         if hasattr(label_id, 'item'):
            label_id = label_id.item()
         predicted_label = id_to_label[label_id]
         print(f"👉 判定結果: [{predicted_label}]")

    print("-" * 30)

🔒 智慧引導安全修補線：配置成功！
🚀 啟動階段四：SetFit 意圖解纏與模型微調 (全遠端智慧對齊黑客版)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



正在從遠端 GitHub 載入訓練資料集...

📊 [降採樣前] 混合訓練資料分佈：
snorkel_label
借貸融資       1945
正常困難負樣本    1563
黃色與特種行業     403
博弈          184
詐騙高風險招募      72
Name: count, dtype: int64

🎯 [降採樣後] SetFit 訓練分佈 (上限 150 筆/類別)：
snorkel_label
正常困難負樣本    150
博弈         150
黃色與特種行業    150
借貸融資       150
詐騙高風險招募     72
Name: count, dtype: int64

訓練類別映射表: {'借貸融資': 0, '博弈': 1, '正常困難負樣本': 2, '詐騙高風險招募': 3, '黃色與特種行業': 4}

正在載入多語系基礎模型，使用加速裝置: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.u

Map:   0%|          | 0/672 [00:00<?, ? examples/s]

...

🎉 強化版模型已儲存至: ./my_intent_classifier_setfit


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())



📊 黃金樣本最終測試成績 (Accuracy): 0.9921

--- 🧠 實戰預測測試 (驗證防禦效果) ---
原始: 日本東京灣的萬豪酒店住宿體驗分享


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow(

KeyError: '黃色與特種行業'

In [2]:
# ==========================================
# 🌟 實戰預測測試修正版 (免重跑訓練，直接載入已儲存的模型)
# ==========================================
import pandas as pd
import urllib.request
import yaml
import re
from setfit import SetFitModel

print("🧠 正在載入剛剛微調完成的本地強化版模型...")
# 直接讀取你花 525 秒修煉成功的精銳模型
model = SetFitModel.from_pretrained("./my_intent_classifier_setfit")

# 1. 載入即時推論的防禦設定 (地名與豁免詞)
YAML_URL = "https://raw.githubusercontent.com/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/main/training%20data/keywords.yaml"
IGNORE_URL = "https://raw.githubusercontent.com/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/main/training%20data/ignore.txt"

def get_sanitizer(yaml_url):
    response = urllib.request.urlopen(yaml_url)
    config = yaml.yaml_response = yaml.safe_load(response)
    loc_config = config.get('locations', {})
    all_locations = []
    for cat in loc_config:
        all_locations.extend(loc_config[cat])
    if not all_locations: return None
    return re.compile(f"({'|'.join(all_locations)})")

city_pattern = get_sanitizer(YAML_URL)

def preprocess_texts(texts, replacement="[LOC]"):
    if not city_pattern: return texts
    if isinstance(texts, str): return city_pattern.sub(replacement, texts)
    return [city_pattern.sub(replacement, str(t)) if pd.notna(t) else "" for t in texts]

ignore_response = urllib.request.urlopen(IGNORE_URL)
ignore_keywords = [line.decode('utf-8').strip() for line in ignore_response if line.strip() and not line.decode('utf-8').startswith('#')]


# 2. 啟動實戰預測
print("\n--- 🧠 實戰預測測試 (驗證防禦效果) ---")

raw_test_sentences = [
    "日本東京灣的萬豪酒店住宿體驗分享",                # 以前會被誤殺成色情的正常樣本
    "全額贷，無勞保也可強力過件，當日放款",            # 借貸融資
    "群組受害者自救會，律師免費諮詢幫你拿回資金",       # 正常困難負樣本
    "註冊首存送 1000，今晚百家樂帶你飛",               # 博弈
    "誠徵台北外派公關，免經驗日領萬元",                 # 黃色特種 (地名脫敏測試)
    "台南信貸，手續簡便當天撥款",                       # 借貸融資 (地名脫敏測試)
    "八大"                                             # 博弈/色情邊界樣本
]

sanitized_sentences = preprocess_texts(raw_test_sentences)

for raw, clean in zip(raw_test_sentences, sanitized_sentences):
    print(f"原始: {raw}")
    if raw != clean:
        print(f"清洗: {clean}")

    # 🛡️ 第一道防線：硬性白名單攔截
    if any(ignore_word in raw for ignore_word in ignore_keywords):
         print(f"👉 判定結果: [正常困難負樣本] (因觸發豁免詞彙)")
    else:
         # 🤖 第二道防線：🚨 修正核心：SetFit 預測直接回傳文字標籤，不需再查表！
         predicted_label = model.predict([clean])[0]
         print(f"👉 判定結果: [{predicted_label}]")

    print("-" * 30)

🧠 正在載入剛剛微調完成的本地強化版模型...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())



--- 🧠 實戰預測測試 (驗證防禦效果) ---
原始: 日本東京灣的萬豪酒店住宿體驗分享
👉 判定結果: [黃色與特種行業]
------------------------------
原始: 全額贷，無勞保也可強力過件，當日放款
👉 判定結果: [借貸融資]
------------------------------
原始: 群組受害者自救會，律師免費諮詢幫你拿回資金
👉 判定結果: [正常困難負樣本] (因觸發豁免詞彙)
------------------------------
原始: 註冊首存送 1000，今晚百家樂帶你飛
👉 判定結果: [正常困難負樣本]
------------------------------
原始: 誠徵台北外派公關，免經驗日領萬元
清洗: 誠徵[LOC]外派公關，免經驗日領萬元
👉 判定結果: [黃色與特種行業]
------------------------------
原始: 台南信貸，手續簡便當天撥款
清洗: [LOC]信貸，手續簡便當天撥款
👉 判定結果: [借貸融資]
------------------------------
原始: 八大
👉 判定結果: [正常困難負樣本]
------------------------------


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datet

In [4]:
import os
# 💡 確保剛才的隨機記錄工具不會在背景干擾上傳
os.environ["WANDB_DISABLED"] = "true"

from huggingface_hub import notebook_login
from setfit import SetFitModel

print("🔑 步驟 1：請進行 Hugging Face 帳號認證")
print("(請在下方欄位貼上你剛剛申請、具備 'Write' 權限的 HF Token)")
# 執行這一行會跳出 Token 輸入框
notebook_login()

🔑 步驟 1：請進行 Hugging Face 帳號認證
(請在下方欄位貼上你剛剛申請、具備 'Write' 權限的 HF Token)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())


In [5]:
print("\n📦 步驟 2：載入你剛剛花 525 秒訓練成功的本地模型...")
local_model_path = "./my_intent_classifier_setfit"
model = SetFitModel.from_pretrained(local_model_path)

# 🚨 請在這裡修改成你的真實 Hugging Face 使用者名稱！
# 例如：如果你的 HF 網址是 huggingface.co/username，這裡就填 "username"
HF_USERNAME = "Alyssalai"
MODEL_NAME = "my_intent_classifier_setfit"

repo_id = f"{HF_USERNAME}/{MODEL_NAME}"

print(f"\n🚀 步驟 3：正在將模型上傳至 Hugging Face Hub -> {repo_id} ...")
# SetFit 內建非常方便的 push_to_hub 函式，會自動幫你建立 Repository 並上傳所有權存檔
model.push_to_hub(repo_id=repo_id)

print(f"\n🎉 大功告成！你的模型已經成功雲端備份。")
print(f"🔗 你的模型專屬網址為: https://huggingface.co/{repo_id}")


📦 步驟 2：載入你剛剛花 525 秒訓練成功的本地模型...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/py


🚀 步驟 3：正在將模型上傳至 Hugging Face Hub -> Alyssalai/my_intent_classifier_setfit ...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ier_setfit/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

  ..._setfit/model.safetensors:   0%|          |  551kB /  471MB            

  ...ier_setfit/model_head.pkl:  81%|########1 | 13.2kB / 16.3kB            


🎉 大功告成！你的模型已經成功雲端備份。
🔗 你的模型專屬網址為: https://huggingface.co/Alyssalai/my_intent_classifier_setfit
